# 02 — Audio Feature Extraction

In [1]:
import librosa
import numpy as np
import pandas as pd
import pyloudnorm as pyln
from pathlib import Path

In [2]:
DATA_DIR = Path("../data/raw/audio")

GENRE_CORPUS_MAP = {
    "compas": "caribbean",
    "zouk": "caribbean",
    "reggae_dancehall": "caribbean",
    "calypso_soca": "caribbean",
    "highlife": "westafrica",
    "juju": "westafrica",
    "kora": "westafrica",
    "afrobeats": "westafrica",
}

TARGET_SR = 22050
MONO = True
TARGET_DURATION = 30
TARGET_LENGTH = TARGET_SR * TARGET_DURATION

KS_MAJOR = np.array([6.35, 2.23, 3.48, 2.33, 4.38, 4.09, 2.52, 5.19, 2.39, 3.66, 2.29, 2.88])
KS_MINOR = np.array([6.33, 2.68, 3.52, 5.38, 2.60, 3.53, 2.54, 4.75, 3.98, 2.69, 3.34, 3.17])
PITCH_CLASSES = ["C", "C#", "D", "D#", "E", "F", "F#", "G", "G#", "A", "A#", "B"]

In [3]:
def estimate_key_krumhansl(chroma_mean):
    best_corr = -np.inf
    best_key, best_mode = None, None
    for i in range(12):
        major_rot = np.roll(KS_MAJOR, i)
        minor_rot = np.roll(KS_MINOR, i)
        corr_major = np.corrcoef(chroma_mean, major_rot)[0, 1]
        corr_minor = np.corrcoef(chroma_mean, minor_rot)[0, 1]
        if corr_major > best_corr:
            best_corr, best_key, best_mode = corr_major, PITCH_CLASSES[i], "major"
        if corr_minor > best_corr:
            best_corr, best_key, best_mode = corr_minor, PITCH_CLASSES[i], "minor"
    return best_key, best_mode, best_corr

In [4]:
def extract_features(filepath):
    y, sr = librosa.load(filepath, sr=TARGET_SR, mono=MONO, duration=TARGET_DURATION)

    if len(y) < TARGET_LENGTH:
        y = np.pad(y, (0, TARGET_LENGTH - len(y)))
    else:
        y = y[:TARGET_LENGTH]

    features = {}

    chroma = librosa.feature.chroma_cqt(y=y, sr=sr)
    chroma_mean = chroma.mean(axis=1)
    features["chroma_mean"] = chroma_mean.tolist()

    key, mode, corr = estimate_key_krumhansl(chroma_mean)
    features["estimated_key"] = key
    features["mode"] = mode
    features["key_correlation"] = float(corr)

    onset_env = librosa.onset.onset_strength(y=y, sr=sr)
    tempo, beat_frames = librosa.beat.beat_track(onset_envelope=onset_env, sr=sr)
    features["tempo_bpm"] = float(np.atleast_1d(tempo)[0])

    beat_times = librosa.frames_to_time(beat_frames, sr=sr)
    features["beat_count"] = int(len(beat_times))

    ibis = np.diff(beat_times)
    features["mean_ibi_sec"] = float(np.mean(ibis))
    features["ibi_std_sec"] = float(np.std(ibis))

    beat_strength = onset_env[beat_frames]
    features["mean_beat_strength"] = float(np.mean(beat_strength))

    rms = librosa.feature.rms(y=y)[0]
    features["rms_mean"] = float(np.mean(rms))
    features["rms_std"] = float(np.std(rms))

    meter = pyln.Meter(sr)
    loudness_lufs = meter.integrated_loudness(y.astype(np.float64))
    features["lufs_integrated"] = float(loudness_lufs)

    peak_db = 20 * np.log10(np.max(np.abs(y)) + 1e-9)
    rms_db_min = 20 * np.log10(np.min(rms[rms > 0]) + 1e-9)
    features["dynamic_range_db"] = float(peak_db - rms_db_min)

    spec_centroid = librosa.feature.spectral_centroid(y=y, sr=sr)[0]
    features["spectral_centroid_mean"] = float(np.mean(spec_centroid))

    spec_bw = librosa.feature.spectral_bandwidth(y=y, sr=sr)[0]
    features["spectral_bandwidth_mean"] = float(np.mean(spec_bw))

    spec_flux = librosa.onset.onset_strength(y=y, sr=sr, feature=librosa.feature.melspectrogram)
    features["spectral_flux_mean"] = float(np.mean(spec_flux))

    onsets = librosa.onset.onset_detect(y=y, sr=sr, units="time")
    first_onset_sample = int(onsets[0] * sr)
    window = y[first_onset_sample : first_onset_sample + int(0.3 * sr)]
    peak_idx = np.argmax(np.abs(window))
    features["log_attack_time_ms"] = float((peak_idx / sr) * 1000)

    mfccs = librosa.feature.mfcc(y=y, sr=sr, n_mfcc=13)
    for i in range(1, 14):
        features[f"mfcc_{i}_mean"] = float(np.mean(mfccs[i - 1]))

    onset_times = librosa.onset.onset_detect(y=y, sr=sr, units="time")
    features["onsets_per_second"] = float(len(onset_times) / TARGET_DURATION)

    spec_flatness = librosa.feature.spectral_flatness(y=y)[0]
    features["textural_grain_mean"] = float(np.mean(spec_flatness))

    chroma_for_ssm = librosa.feature.chroma_cqt(y=y, sr=sr)
    ssm = librosa.segment.recurrence_matrix(chroma_for_ssm, mode="affinity", sym=True)
    features["ssm_shape"] = list(ssm.shape)
    features["ssm_mean_affinity"] = float(np.mean(ssm))

    bounds = librosa.segment.agglomerative(chroma_for_ssm, k=5)
    bound_times = librosa.frames_to_time(bounds, sr=sr)
    features["segment_boundary_times"] = bound_times.tolist()
    features["segment_count"] = int(len(bound_times))

    return features

In [5]:
records = []

for genre, corpus in GENRE_CORPUS_MAP.items():
    genre_dir = DATA_DIR / genre
    if not genre_dir.exists():
        continue

    for file_path in genre_dir.iterdir():
        if file_path.suffix.lower() in [".wav", ".mp3", ".m4a"]:
            row = {
                "track_id": file_path.stem,
                "genre": genre,
                "corpus": corpus,
                "file_path": str(file_path),
            }
            row.update(extract_features(file_path))
            records.append(row)

features_df = pd.DataFrame(records)

In [6]:
OUTPUT_PATH = Path("../data/processed/features.csv")
OUTPUT_PATH.parent.mkdir(parents=True, exist_ok=True)
features_df.to_csv(OUTPUT_PATH, index=False)